In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC datetimes (reuse variable names for efficiency)
df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

df_log = pd.read_csv("log.csv")
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

df_error = pd.read_csv("error_logs.csv")
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")

# Helper to aggregate per-group stats with global P95 and window counts/min/max
def aggregate_with_window(df, name_col, value_col='value', top_n=50):
    # Global aggregations per group (global means use full series)
    gb = df.groupby(['cmdb_id', name_col])
    count_total = gb.size().rename('count_total')
    p95 = gb[value_col].quantile(0.95).rename('p95_value')
    summary = pd.concat([count_total, p95], axis=1).reset_index()
    # Window-limited slice and its per-group stats
    df_win = df[(df['timestamp'] >= window_start) & (df['timestamp'] <= window_end)]
    if not df_win.empty:
        gb_win = df_win.groupby(['cmdb_id', name_col])
        count_in_window = gb_win.size().rename('count_in_window')
        min_ts = gb_win['timestamp'].min().rename('min_ts_in_window')
        max_ts = gb_win['timestamp'].max().rename('max_ts_in_window')
        win_summary = pd.concat([count_in_window, min_ts, max_ts], axis=1).reset_index()
    else:
        # empty structure with proper columns
        win_summary = pd.DataFrame(columns=['cmdb_id', name_col, 'count_in_window', 'min_ts_in_window', 'max_ts_in_window'])
    # Merge global and window summaries
    merged = summary.merge(win_summary, on=['cmdb_id', name_col], how='left')
    # Fill missing window counts with 0 and keep NaT for min/max
    merged['count_in_window'] = merged['count_in_window'].fillna(0).astype(int)
    # Sort and limit
    merged = merged.sort_values('count_total', ascending=False).head(top_n)
    # Ensure column order
    merged = merged[['cmdb_id', name_col, 'count_total', 'p95_value', 'count_in_window', 'min_ts_in_window', 'max_ts_in_window']]
    return merged

# Aggregate metric.csv by (cmdb_id, kpi_name)
metric_summary = aggregate_with_window(df_metric, 'kpi_name', 'value', top_n=50)

# Aggregate trace.csv by (cmdb_id, trace_name)
trace_summary = aggregate_with_window(df_trace, 'trace_name', 'value', top_n=50)

# Aggregate log.csv by (cmdb_id, log_name)
log_summary = aggregate_with_window(df_log, 'log_name', 'value', top_n=50)

# Error logs overall counts and samples inside window (up to 5)
error_total = len(df_error)
df_error_win = df_error[(df_error['timestamp'] >= window_start) & (df_error['timestamp'] <= window_end)]
error_in_window_count = len(df_error_win)
# sample up to 5 rows with selected columns
error_samples = df_error_win[['timestamp', 'cmdb_id', 'message']].head(5)

# Display results compactly (variables to be shown as output)
metric_summary, trace_summary, log_summary, error_total, error_in_window_count, error_samples

```
Out[1]:
```
# Compose a plain-English summary based on the previously computed summaries (reuse variables)
m = metric_summary
t = trace_summary
l = log_summary
etot = error_total
ewin = error_in_window_count

# compute share of top groups where all points fall inside the incident window
m_fullshare = (m['count_in_window'] == m['count_total']).mean() if not m.empty else 0.0
t_fullshare = (t['count_in_window'] == t['count_total']).mean() if not t.empty else 0.0
l_fullshare = (l['count_in_window'] == l['count_total']).mean() if not l.empty else 0.0

# pick examples (safe access)
def example_row(df):
    if df.empty:
        return None
    r = df.iloc[0]
    return f"{r['cmdb_id']} / {r[df.columns[1]]} (count_total={r['count_total']}, p95={r['p95_value']})"

m_ex = example_row(m)
t_ex = example_row(t)
l_ex = example_row(l)

summary = (
    "Summary of telemetry for incident window 2024-01-23 06:25:46 to 2024-01-23 06:55:46 UTC:\n\n"
    f"- Metrics: Top groups (up to 50) were computed. {int(m_fullshare*100)}% of those top metric groups "
    "have all their samples inside the incident window (most groups show count_total=25 and count_in_window=25). "
    f"Example metric: {m_ex}.\n\n"
    f"- Traces: Top trace groups similarly show full coverage in the window for {int(t_fullshare*100)}% of the top groups. "
    f"Example trace: {t_ex}.\n\n"
    f"- Logs: Top log groups also largely fall fully inside the window ({int(l_fullshare*100)}% of top groups). "
    f"Example log: {l_ex}.\n\n"
    "- Error logs: the error_logs dataset contains 0 rows overall and 0 rows inside the incident window; no samples to show.\n\n"
    "Conclusion: Metric, trace, and log telemetry for many services were present and fully cover the incident window (many groups show 25 samples with p95 values shown). "
    "There are no error log entries in the provided error_logs.csv for the incident window."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count_total     p95_value  count_in_window          min_ts_in_window          max_ts_in_window
374           ts-voucher-service      socket           25  4.000000e+00               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
0    ts-admin-basic-info-service         cpu           25  4.600729e-01               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
1    ts-admin-basic-info-service      diskio           25  5.794424e+04               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
2    ts-admin-basic-info-service  latency-50           25  1.695932e-02               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
3    ts-admin-basic-info-service  latency-90           25  2.339186e-02               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
..                           ...         ...          ...           ...              ...                       ...                       ...
325            ts-travel-service      diskio           25  5.895775e+04               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
326            ts-travel-service       error           25  0.000000e+00               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
311             ts-train-service  latency-50           25  7.150380e-03               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
312             ts-train-service  latency-90           25  9.596971e-03               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
313             ts-train-service         mem           25  2.542752e+08               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00

[50 rows x 7 columns],                 cmdb_id                                         trace_name  count_total    p95_value  count_in_window          min_ts_in_window          max_ts_in_window
791     ts-user-service                 trace.to_ts-user-service.row_count           25    17.600000               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
0                  root  trace.to_ts-admin-basic-info-service.duration_...           25     0.010971               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
1                  root  trace.to_ts-admin-basic-info-service.duration_p95           25     0.017040               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
2                  root    trace.to_ts-admin-basic-info-service.error_rate           25     0.000000               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
3                  root     trace.to_ts-admin-basic-info-service.row_count           25    20.000000               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
..                  ...                                                ...          ...          ...              ...                       ...                       ...
729   ts-travel-service            trace.to_ts-travel-service.duration_p95           25     0.119691               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
730   ts-travel-service              trace.to_ts-travel-service.error_rate           25     0.000000               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
731   ts-travel-service               trace.to_ts-travel-service.row_count           25  1605.800000               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
732  ts-travel2-service                      trace.from_root.duration_mean           25     0.869742               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
733  ts-travel2-service                       trace.from_root.duration_p95           25     1.093775               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00

[50 rows x 7 columns],                         cmdb_id         log_name  count_total  p95_value  count_in_window          min_ts_in_window          max_ts_in_window
0   ts-admin-basic-info-service  log.error_count           25        0.0               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
1   ts-admin-basic-info-service    log.row_count           25       30.0               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
2       ts-admin-travel-service  log.error_count           25        0.0               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
3       ts-admin-travel-service    log.row_count           25       30.0               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
7          ts-assurance-service    log.row_count           25       12.8               25 2024-01-23 06:28:00+00:00 2024-01-23 06:52:00+00:00
..                          ...              ...          ...        ...              ...                       ...                       ...
18       ts-consign-price-mongo  log.error_count           24        0.0               24 2024-01-23 06:29:00+00:00 2024-01-23 06:52:00+00:00
26            ts-food-map-mongo  log.error_count           24        0.0               24 2024-01-23 06:29:00+00:00 2024-01-23 06:52:00+00:00
27            ts-food-map-mongo    log.row_count           24        1.0               24 2024-01-23 06:29:00+00:00 2024-01-23 06:52:00+00:00
23            ts-contacts-mongo    log.row_count           24        1.0               24 2024-01-23 06:29:00+00:00 2024-01-23 06:52:00+00:00
22            ts-contacts-mongo  log.error_count           24        0.0               24 2024-01-23 06:29:00+00:00 2024-01-23 06:52:00+00:00

[50 rows x 7 columns], 0, 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse df_metric if present, else load
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp parsed
    if not pd.api.types.is_datetime64_any_dtype(df_metric['timestamp']):
        df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")

# Candidate services
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Filter metric data to candidate services (for threshold calc we still use full available series of these services)
df_cand = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# Compute global thresholds (P95 and P5) and total count per (cmdb_id, kpi_name) using full series (rule 9)
gb_all = df_cand.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    count_total='count',
    p95_value=lambda x: x.quantile(0.95),
    p5_value=lambda x: x.quantile(0.05)
).reset_index()

# Windowed points for candidate services
df_win = df_cand[(df_cand['timestamp'] >= window_start) & (df_cand['timestamp'] <= window_end)].copy()

# Aggregate window stats per group
if not df_win.empty:
    win_stats = df_win.groupby(['cmdb_id', 'kpi_name']).agg(
        count_in_window=('value','count'),
        min_in_window=('value','min'),
        max_in_window=('value','max'),
        earliest_ts_in_window=('timestamp','min')
    ).reset_index()
else:
    win_stats = pd.DataFrame(columns=['cmdb_id','kpi_name','count_in_window','min_in_window','max_in_window','earliest_ts_in_window'])

# Prepare anomaly flags by merging thresholds into window rows
if not df_win.empty:
    df_win_thr = df_win.merge(gb_all, on=['cmdb_id','kpi_name'], how='left')
    df_win_thr['is_high'] = df_win_thr['value'] > df_win_thr['p95_value']
    df_win_thr['is_low'] = df_win_thr['value'] <= df_win_thr['p5_value']

    high_agg = df_win_thr[df_win_thr['is_high']].groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count_high=('is_high','sum'),
        earliest_anom_high=('timestamp','min')
    ).reset_index()

    low_agg = df_win_thr[df_win_thr['is_low']].groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count_low=('is_low','sum'),
        earliest_anom_low=('timestamp','min')
    ).reset_index()
else:
    high_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count_high','earliest_anom_high'])
    low_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count_low','earliest_anom_low'])

# Merge all pieces: global thresholds + window stats + anomaly aggregates
result = gb_all.merge(win_stats, on=['cmdb_id','kpi_name'], how='left') \
               .merge(high_agg, on=['cmdb_id','kpi_name'], how='left') \
               .merge(low_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill missing numeric window/anomaly counts with 0, keep timestamp NaT where missing
result['count_in_window'] = result['count_in_window'].fillna(0).astype(int)
result['min_in_window'] = result['min_in_window']  # may be NaN
result['max_in_window'] = result['max_in_window']
result['earliest_ts_in_window'] = pd.to_datetime(result['earliest_ts_in_window'])
result['anomaly_count_high'] = result['anomaly_count_high'].fillna(0).astype(int)
result['anomaly_count_low'] = result['anomaly_count_low'].fillna(0).astype(int)
result['earliest_anom_high'] = pd.to_datetime(result['earliest_anom_high'])
result['earliest_anom_low'] = pd.to_datetime(result['earliest_anom_low'])

# Order columns as requested
result_table = result[['cmdb_id','kpi_name','count_total','p95_value','p5_value',
                       'count_in_window','min_in_window','max_in_window','earliest_ts_in_window',
                       'anomaly_count_high','earliest_anom_high','anomaly_count_low','earliest_anom_low']]

# Sort by count_total desc then anomaly_count_high desc, limit to top 50
result_table = result_table.sort_values(['count_total','anomaly_count_high'], ascending=[False, False]).head(50)

# Display compact result
result_table

```
Out[2]:
```
# Reuse the previously computed result_table to build a plain-English summary string
rt = result_table.copy()

n_groups = rt.shape[0]
percent_full_window = (rt['count_in_window'] == rt['count_total']).mean() * 100

high_df = rt[rt['anomaly_count_high'] > 0].sort_values(['anomaly_count_high','count_total'], ascending=[False, False])
n_high_groups = high_df.shape[0]

# Aggregate high-anomaly counts by service and earliest high anomaly per service
high_by_service = high_df.groupby('cmdb_id').agg(
    total_high_anomalies=('anomaly_count_high','sum'),
    earliest_high=('earliest_anom_high','min')
).reset_index().sort_values('total_high_anomalies', ascending=False)

# Convert earliest timestamps to ISO strings for clear display
high_by_service['earliest_high'] = high_by_service['earliest_high'].astype(str).replace('NaT', 'none')

# Note on low anomalies for 'error' KPIs
error_rows = rt[rt['kpi_name'] == 'error']
error_all_low = error_rows[error_rows['anomaly_count_low'] == error_rows['count_in_window']].shape[0]

summary = (
    f"Summary for candidate services (ts-auth/order/route/train/travel):\n\n"
    f"- Groups analyzed: {n_groups} metric (cmdb_id, kpi_name) groups.\n"
    f"- Coverage: {percent_full_window:.0f}% of these groups have all their samples inside the incident window (count_in_window == count_total).\n\n"
    f"- High anomalies: {n_high_groups} groups show at least one in-window value above the global P95.\n"
)

# Add a short list of services most affected by high anomalies (top 5)
if not high_by_service.empty:
    top_services = ", ".join(
        f"{row['cmdb_id']} (high anomalies={row['total_high_anomalies']}, earliest={row['earliest_high']})"
        for _, row in high_by_service.head(5).iterrows()
    )
    summary += f"- Most affected services (by total high anomalies): {top_services}.\n"
else:
    summary += "- No high anomalies detected in the candidate groups.\n"

# Note about error KPIs
summary += (
    f"\n- Error KPIs: {error_rows.shape[0]} 'error' metric groups present for the candidates; "
    f"{error_all_low} of them report all in-window points at or below the P5 (often P5=0), indicating error rates stayed at zero/very low during the window.\n\n"
    "Key observations:\n"
    "- ts-auth-service exhibits multiple high anomalies across CPU, disk I/O, latency and memory-related KPIs (each with a small number of in-window points above P95, earliest highs around 06:28–06:32 UTC).\n"
    "- Other candidate services (ts-travel-service and others) also show spikes for workload/trace-related KPIs, but error metrics remain low/zero.\n\n"
    "Conclusion: Several candidate services — notably ts-auth-service — experienced short high-value spikes (above their global P95) during the incident window, while error rates remained low. "
    "These high spikes are candidates for further investigation (check the specific KPI times shown in the detailed table if needed)."
)

# Display the summary string
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count_total     p95_value      p5_value  count_in_window  min_in_window  max_in_window     earliest_ts_in_window  anomaly_count_high        earliest_anom_high  anomaly_count_low         earliest_anom_low
0     ts-auth-service         cpu           25  1.532888e+01  1.290796e+01               25   1.232102e+01   1.555749e+01 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:29:00+00:00                  2 2024-01-23 06:42:00+00:00
1     ts-auth-service      diskio           25  6.027503e+04  3.393843e+04               25   3.108852e+04   6.270858e+04 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:32:00+00:00                  2 2024-01-23 06:33:00+00:00
3     ts-auth-service  latency-50           25  2.774159e-01  2.168744e-01               25   2.113033e-01   2.836245e-01 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:29:00+00:00                  2 2024-01-23 06:41:00+00:00
4     ts-auth-service  latency-90           25  8.767270e-01  4.769345e-01               25   4.531635e-01   9.269075e-01 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:31:00+00:00                  2 2024-01-23 06:37:00+00:00
5     ts-auth-service         mem           25  2.453703e+08  2.453317e+08               25   2.453256e+08   2.454411e+08 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:28:00+00:00                  2 2024-01-23 06:30:00+00:00
..                ...         ...          ...           ...           ...              ...            ...            ...                       ...                 ...                       ...                ...                       ...
38  ts-travel-service    workload           25  1.000051e+01  6.754835e+00               25   5.910117e+00   1.060760e+01 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:31:00+00:00                  2 2024-01-23 06:28:00+00:00
33  ts-travel-service       error           25  0.000000e+00  0.000000e+00               25   0.000000e+00   3.336667e-02 2024-01-23 06:28:00+00:00                   1 2024-01-23 06:37:00+00:00                 24 2024-01-23 06:28:00+00:00
2     ts-auth-service       error           25  0.000000e+00  0.000000e+00               25   0.000000e+00   0.000000e+00 2024-01-23 06:28:00+00:00                   0                       NaT                 25 2024-01-23 06:28:00+00:00
10   ts-order-service       error           25  0.000000e+00  0.000000e+00               25   0.000000e+00   0.000000e+00 2024-01-23 06:28:00+00:00                   0                       NaT                 25 2024-01-23 06:28:00+00:00
18   ts-route-service       error           25  0.000000e+00  0.000000e+00               25   0.000000e+00   0.000000e+00 2024-01-23 06:28:00+00:00                   0                       NaT                 25 2024-01-23 06:28:00+00:00

[39 rows x 13 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse df_trace if present, else load and parse timestamps
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")

# Candidate services + include 'root' traces
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']
include_ids = candidates + ['root']

# Restrict trace rows to candidates and root
df_trace_cand = df_trace[df_trace['cmdb_id'].isin(include_ids)].copy()

# Global thresholds (P95 and P5) and total count per (cmdb_id, trace_name) using full series
gb_all = df_trace_cand.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    count_total='count',
    p95_value=lambda x: x.quantile(0.95),
    p5_value=lambda x: x.quantile(0.05)
).reset_index()

# Windowed rows
df_trace_win = df_trace_cand[(df_trace_cand['timestamp'] >= window_start) & (df_trace_cand['timestamp'] <= window_end)].copy()

# Window aggregates per group
if not df_trace_win.empty:
    win_stats = df_trace_win.groupby(['cmdb_id','trace_name']).agg(
        count_in_window=('value','count'),
        min_in_window=('value','min'),
        max_in_window=('value','max'),
        earliest_ts_in_window=('timestamp','min')
    ).reset_index()
else:
    win_stats = pd.DataFrame(columns=['cmdb_id','trace_name','count_in_window','min_in_window','max_in_window','earliest_ts_in_window'])

# Merge thresholds into window rows to compute anomalies
if not df_trace_win.empty:
    df_win_thr = df_trace_win.merge(gb_all, on=['cmdb_id','trace_name'], how='left')
    df_win_thr['is_high'] = df_win_thr['value'] > df_win_thr['p95_value']
    df_win_thr['is_low'] = df_win_thr['value'] <= df_win_thr['p5_value']

    high_agg = df_win_thr[df_win_thr['is_high']].groupby(['cmdb_id','trace_name']).agg(
        anomaly_count_high=('is_high','sum'),
        earliest_anom_high=('timestamp','min')
    ).reset_index()

    low_agg = df_win_thr[df_win_thr['is_low']].groupby(['cmdb_id','trace_name']).agg(
        anomaly_count_low=('is_low','sum'),
        earliest_anom_low=('timestamp','min')
    ).reset_index()
else:
    high_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count_high','earliest_anom_high'])
    low_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count_low','earliest_anom_low'])

# Merge all pieces
trace_result = gb_all.merge(win_stats, on=['cmdb_id','trace_name'], how='left') \
                     .merge(high_agg, on=['cmdb_id','trace_name'], how='left') \
                     .merge(low_agg, on=['cmdb_id','trace_name'], how='left')

# Fill missing numeric window/anomaly counts with 0, keep timestamps as NaT where missing
trace_result['count_in_window'] = trace_result['count_in_window'].fillna(0).astype(int)
trace_result['anomaly_count_high'] = trace_result['anomaly_count_high'].fillna(0).astype(int)
trace_result['anomaly_count_low'] = trace_result['anomaly_count_low'].fillna(0).astype(int)

trace_result['earliest_ts_in_window'] = pd.to_datetime(trace_result['earliest_ts_in_window'])
trace_result['earliest_anom_high'] = pd.to_datetime(trace_result['earliest_anom_high'])
trace_result['earliest_anom_low'] = pd.to_datetime(trace_result['earliest_anom_low'])

# Select and order columns as requested
trace_result_table = trace_result[['cmdb_id','trace_name','count_total','p95_value','p5_value',
                                   'count_in_window','min_in_window','max_in_window','earliest_ts_in_window',
                                   'anomaly_count_high','earliest_anom_high','anomaly_count_low','earliest_anom_low']]

# Sort by anomaly_count_high desc then count_total desc and limit to top 20
trace_result_top20 = trace_result_table.sort_values(['anomaly_count_high','count_total'], ascending=[False, False]).head(20)

# Display compact result
trace_result_top20

```
Out[3]:
```
# Build a concise plain-English summary from the computed trace_result_top20
rt = trace_result_top20

n_groups = rt.shape[0]
n_high = int((rt['anomaly_count_high'] > 0).sum())
earliest_high_ts = rt['earliest_anom_high'].min()
earliest_high_str = str(earliest_high_ts) if not pd.isna(earliest_high_ts) else "none"

# top 3 trace groups by anomaly_count_high (tie-broken by count_total)
top3 = rt.sort_values(['anomaly_count_high','count_total'], ascending=[False, False]).head(3)
top3_list = [
    f"{row['cmdb_id']} / {row['trace_name']} (highs={int(row['anomaly_count_high'])}, earliest={row['earliest_anom_high']})"
    for _, row in top3.iterrows()
]

summary = (
    f"Trace analysis summary for candidate services + 'root' (window 2024-01-23 06:25:46–06:55:46 UTC):\n\n"
    f"- Examined top {n_groups} (cmdb_id, trace_name) groups.\n"
    f"- {n_high} of these groups had at least one in-window value above their global P95.\n"
    f"- The earliest high anomaly observed among these top groups is: {earliest_high_str} UTC.\n\n"
    f"- Top affected trace groups (by high-anomaly count):\n  • " + "\n  • ".join(top3_list) + "\n\n"
    f"Key observation: Many cross-service ('root') traces show short spikes (typically 2 high samples per group) during the incident window across multiple target services (including traces to ts-auth-service, admin services, assurance, etc.). "
    "These spikes are brief and co-occur with low-valued points in the same window (some groups also show values <= P5). Investigate the listed top trace groups and their timestamps for root-cause digging (start with the top entries shown)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         trace_name  count_total  p95_value   p5_value  count_in_window  min_in_window  max_in_window     earliest_ts_in_window  anomaly_count_high        earliest_anom_high  anomaly_count_low         earliest_anom_low
0     root  trace.to_ts-admin-basic-info-service.duration_...           25   0.010971   0.009821               25       0.009716       0.011084 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:37:00+00:00                  2 2024-01-23 06:38:00+00:00
1     root  trace.to_ts-admin-basic-info-service.duration_p95           25   0.017040   0.011119               25       0.010347       0.018230 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:37:00+00:00                  2 2024-01-23 06:28:00+00:00
4     root     trace.to_ts-admin-travel-service.duration_mean           25   1.070733   0.159961               25       0.159783       1.108189 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:46:00+00:00                  2 2024-01-23 06:28:00+00:00
5     root      trace.to_ts-admin-travel-service.duration_p95           25   1.261906   0.166632               25       0.159819       1.310705 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:45:00+00:00                  2 2024-01-23 06:28:00+00:00
8     root        trace.to_ts-assurance-service.duration_mean           25   0.003150   0.002460               25       0.002451       0.004042 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:31:00+00:00                  2 2024-01-23 06:45:00+00:00
9     root         trace.to_ts-assurance-service.duration_p95           25   0.004212   0.002641               25       0.002556       0.008775 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:31:00+00:00                  2 2024-01-23 06:44:00+00:00
11    root            trace.to_ts-assurance-service.row_count           25   8.800000   2.400000               25       2.000000      10.000000 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:43:00+00:00                  2 2024-01-23 06:28:00+00:00
12    root             trace.to_ts-auth-service.duration_mean           25   0.414734   0.270259               25       0.264351       0.429080 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:31:00+00:00                  2 2024-01-23 06:37:00+00:00
13    root              trace.to_ts-auth-service.duration_p95           25   0.988146   0.489682               25       0.453367       1.226409 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:31:00+00:00                  2 2024-01-23 06:37:00+00:00
15    root                 trace.to_ts-auth-service.row_count           25  82.800000  55.400000               25      17.000000      86.000000 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:29:00+00:00                  2 2024-01-23 06:28:00+00:00
16    root          trace.to_ts-consign-service.duration_mean           25   0.021209   0.018053               25       0.017427       0.026787 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:42:00+00:00                  2 2024-01-23 06:45:00+00:00
17    root           trace.to_ts-consign-service.duration_p95           25   0.027004   0.018940               25       0.018614       0.054005 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:32:00+00:00                  2 2024-01-23 06:28:00+00:00
19    root              trace.to_ts-consign-service.row_count           25   9.800000   2.000000               25       1.000000      12.000000 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:30:00+00:00                  3 2024-01-23 06:28:00+00:00
20    root         trace.to_ts-contacts-service.duration_mean           25   0.006527   0.005100               25       0.004912       0.006598 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:33:00+00:00                  2 2024-01-23 06:48:00+00:00
21    root          trace.to_ts-contacts-service.duration_p95           25   0.011099   0.005591               25       0.005463       0.012503 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:33:00+00:00                  2 2024-01-23 06:28:00+00:00
23    root             trace.to_ts-contacts-service.row_count           25   8.800000   2.400000               25       2.000000      10.000000 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:43:00+00:00                  2 2024-01-23 06:28:00+00:00
24    root             trace.to_ts-food-service.duration_mean           25   0.118378   0.044076               25       0.043835       0.121181 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:41:00+00:00                  2 2024-01-23 06:28:00+00:00
25    root              trace.to_ts-food-service.duration_p95           25   0.209167   0.048345               25       0.045156       0.214558 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:41:00+00:00                  2 2024-01-23 06:28:00+00:00
27    root                 trace.to_ts-food-service.row_count           25  26.400000  12.600000               25       5.000000      27.000000 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:47:00+00:00                  2 2024-01-23 06:28:00+00:00
28    root   trace.to_ts-inside-payment-service.duration_mean           25   0.060266   0.026428               25       0.025930       0.062602 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:30:00+00:00                  2 2024-01-23 06:35:00+00:00```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse loaded DataFrames if available, else load and parse timestamps
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")

# 1) Metrics for cmdb_id == 'ts-auth-service'
cmdb_target = 'ts-auth-service'
df_m_auth = df_metric[df_metric['cmdb_id'] == cmdb_target].copy()

# Global thresholds (using full series for this component, per kpi_name)
if not df_m_auth.empty:
    thresholds_m = df_m_auth.groupby('kpi_name')['value'].agg(
        p95_value=lambda x: x.quantile(0.95),
        p5_value =lambda x: x.quantile(0.05)
    ).reset_index()
else:
    thresholds_m = pd.DataFrame(columns=['kpi_name','p95_value','p5_value'])

# Windowed metric rows for this component
df_m_auth_win = df_m_auth[(df_m_auth['timestamp'] >= window_start) & (df_m_auth['timestamp'] <= window_end)].copy()
# Merge thresholds into window rows
df_m_auth_win = df_m_auth_win.merge(thresholds_m, on='kpi_name', how='left')

# Flagging
def flag_row(val, p95, p5):
    if pd.isna(val) or pd.isna(p95) or pd.isna(p5):
        return 'ok'
    if val > p95:
        return 'high'
    if val <= p5:
        return 'low'
    return 'ok'

if not df_m_auth_win.empty:
    df_m_auth_win['flag'] = df_m_auth_win.apply(lambda r: flag_row(r['value'], r['p95_value'], r['p5_value']), axis=1)
else:
    df_m_auth_win['flag'] = []

# Select requested columns and sort by timestamp ascending, limit 50
metric_auth_rows = df_m_auth_win[['timestamp','kpi_name','value','p95_value','p5_value','flag']].sort_values('timestamp', ascending=True).head(50)

# 2) Traces where trace_name contains 'auth-service' (trace target is ts-auth-service)
df_t_auth = df_trace[df_trace['trace_name'].str.contains('auth-service', na=False)].copy()

# Compute global thresholds per (cmdb_id, trace_name) using full series
if not df_t_auth.empty:
    thresholds_t = df_t_auth.groupby(['cmdb_id','trace_name'])['value'].agg(
        p95_value=lambda x: x.quantile(0.95),
        p5_value =lambda x: x.quantile(0.05)
    ).reset_index()
else:
    thresholds_t = pd.DataFrame(columns=['cmdb_id','trace_name','p95_value','p5_value'])

# Windowed trace rows
df_t_auth_win = df_t_auth[(df_t_auth['timestamp'] >= window_start) & (df_t_auth['timestamp'] <= window_end)].copy()
df_t_auth_win = df_t_auth_win.merge(thresholds_t, on=['cmdb_id','trace_name'], how='left')

# Flagging for traces
if not df_t_auth_win.empty:
    df_t_auth_win['flag'] = df_t_auth_win.apply(lambda r: flag_row(r['value'], r['p95_value'], r['p5_value']), axis=1)
else:
    df_t_auth_win['flag'] = []

trace_auth_rows = df_t_auth_win[['timestamp','trace_name','value','p95_value','p5_value','flag']].sort_values('timestamp', ascending=True).head(50)

# 3) Logs for cmdb_id == 'ts-auth-service' inside the window
df_log_auth_win = df_log[(df_log['cmdb_id'] == cmdb_target) & 
                         (df_log['timestamp'] >= window_start) & (df_log['timestamp'] <= window_end)].copy()

log_auth_rows = df_log_auth_win[['timestamp','log_name','value']].sort_values('timestamp', ascending=True).head(20)

# Return the three compact tables (variables displayed)
metric_auth_rows, trace_auth_rows, log_auth_rows

```
Out[4]:
```
Summary for ts-auth-service during 2024-01-23 06:25:46–06:55:46 UTC:

- Metrics (per-minute):
  - Most metric samples for ts-auth-service fell within their normal range (flagged "ok") during the window in the displayed rows.
  - The "error" metric remained at or below its P5 (value = 0 → flagged "low") throughout the window — there were no error-rate increases.
  - Some metrics (e.g., diskio, others) show values below their P5 at certain timestamps (flagged "low" in the table), but no widespread or sustained metric values above global P95 appear in the shown metric rows.

- Traces:
  - Several auth-related trace metrics show brief high spikes above their global P95 around ~06:31 UTC (examples: trace.to_ts-auth-service.duration_mean, trace.to_ts-auth-service.duration_p95, trace.from_ts-auth-service.duration_p95 — flagged "high" at ~06:31).
  - Other trace fields (row_count) were low earlier (~06:28) and then rose later; some row_count values are below P5 at 06:28 and then approach typical/higher values later in the window.
  - Overall: short-lived latency spikes in traces around ~06:31 are the most notable anomaly signal.

- Logs:
  - log.row_count shows substantial traffic counts during the window (examples: 68 at 06:28, then hundreds at 06:29–06:34, e.g., 342, 332, 330, 320…).
  - log.error_count is 0 for all displayed timestamps (no error log volume).

Conclusion / recommendation:
- The strongest signal is brief trace latency spikes affecting auth-related traces around ~06:31 UTC. Metrics do not show clear sustained CPU/memory overload or increased error rates, and logs show normal/large request volumes but no error entries.
- Next steps: investigate the trace spans around 06:30–06:32 (auth-service and upstream callers), correlate with request volume changes and any upstream service behavior to identify root cause of the transient latency spikes.

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp    kpi_name         value     p95_value      p5_value flag
0  2024-01-23 06:28:00+00:00         cpu  1.409661e+01  1.532888e+01  1.290796e+01   ok
1  2024-01-23 06:28:00+00:00      diskio  4.529130e+04  6.027503e+04  3.393843e+04   ok
2  2024-01-23 06:28:00+00:00       error  0.000000e+00  0.000000e+00  0.000000e+00  low
3  2024-01-23 06:28:00+00:00  latency-50  2.365144e-01  2.774159e-01  2.168744e-01   ok
4  2024-01-23 06:28:00+00:00  latency-90  6.616815e-01  8.767270e-01  4.769345e-01   ok
..                       ...         ...           ...           ...           ...  ...
42 2024-01-23 06:33:00+00:00       error  0.000000e+00  0.000000e+00  0.000000e+00  low
41 2024-01-23 06:33:00+00:00      diskio  3.108852e+04  6.027503e+04  3.393843e+04  low
43 2024-01-23 06:33:00+00:00  latency-50  2.205261e-01  2.774159e-01  2.168744e-01   ok
53 2024-01-23 06:34:00+00:00         mem  2.453588e+08  2.453703e+08  2.453317e+08   ok
55 2024-01-23 06:34:00+00:00    workload  2.713400e+00  2.762010e+00  2.465400e+00   ok

[50 rows x 6 columns],                    timestamp                               trace_name       value   p95_value    p5_value  flag
0  2024-01-23 06:28:00+00:00   trace.to_ts-auth-service.duration_mean    0.281984    0.414734    0.270259    ok
11 2024-01-23 06:28:00+00:00       trace.to_ts-auth-service.row_count   68.000000  331.200000  221.200000   low
10 2024-01-23 06:28:00+00:00      trace.to_ts-auth-service.error_rate    0.000000    0.000000    0.000000   low
9  2024-01-23 06:28:00+00:00    trace.to_ts-auth-service.duration_p95    0.002630    0.006367    0.002244    ok
7  2024-01-23 06:28:00+00:00     trace.from_ts-auth-service.row_count   68.000000  331.200000  221.200000   low
..                       ...                                      ...         ...         ...         ...   ...
41 2024-01-23 06:31:00+00:00  trace.from_ts-auth-service.duration_p95    0.006516    0.006367    0.002244  high
37 2024-01-23 06:31:00+00:00    trace.to_ts-auth-service.duration_p95    1.226409    0.988146    0.489682  high
36 2024-01-23 06:31:00+00:00   trace.to_ts-auth-service.duration_mean    0.415800    0.414734    0.270259  high
59 2024-01-23 06:32:00+00:00       trace.to_ts-auth-service.row_count  320.000000  331.200000  221.200000    ok
58 2024-01-23 06:32:00+00:00      trace.to_ts-auth-service.error_rate    0.000000    0.000000    0.000000   low

[50 rows x 6 columns],                     timestamp         log_name  value
6   2024-01-23 06:28:00+00:00  log.error_count      0
7   2024-01-23 06:28:00+00:00    log.row_count     68
50  2024-01-23 06:29:00+00:00  log.error_count      0
51  2024-01-23 06:29:00+00:00    log.row_count    342
140 2024-01-23 06:30:00+00:00  log.error_count      0
141 2024-01-23 06:30:00+00:00    log.row_count    332
228 2024-01-23 06:31:00+00:00  log.error_count      0
229 2024-01-23 06:31:00+00:00    log.row_count    330
318 2024-01-23 06:32:00+00:00  log.error_count      0
319 2024-01-23 06:32:00+00:00    log.row_count    320
408 2024-01-23 06:33:00+00:00  log.error_count      0
409 2024-01-23 06:33:00+00:00    log.row_count    324
499 2024-01-23 06:34:00+00:00    log.row_count    320
498 2024-01-23 06:34:00+00:00  log.error_count      0
588 2024-01-23 06:35:00+00:00  log.error_count      0
589 2024-01-23 06:35:00+00:00    log.row_count    316
678 2024-01-23 06:36:00+00:00  log.error_count      0
679 2024-01-23 06:36:00+00:00    log.row_count    324
768 2024-01-23 06:37:00+00:00  log.error_count      0
769 2024-01-23 06:37:00+00:00    log.row_count    312)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse df_metric if present, else load and parse timestamp
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")

# Candidate services
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Restrict to candidate services for threshold calculation (use full series per group)
df_cand = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# Compute global thresholds (P95, P5) and total count per (cmdb_id, kpi_name) using full series
thr = df_cand.groupby(['cmdb_id','kpi_name'])['value'].agg(
    count_total='count',
    p95_value=lambda x: x.quantile(0.95),
    p5_value =lambda x: x.quantile(0.05)
).reset_index()

# Filter points to the incident window
df_win = df_cand[(df_cand['timestamp'] >= window_start) & (df_cand['timestamp'] <= window_end)].copy()

# Merge thresholds into window rows to mark high anomalies (value > p95)
if not df_win.empty:
    df_win = df_win.merge(thr[['cmdb_id','kpi_name','p95_value','p5_value']], on=['cmdb_id','kpi_name'], how='left')
    df_win['is_high'] = (df_win['value'] > df_win['p95_value']).astype(int)
else:
    # create empty consistent frame
    df_win = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','p95_value','p5_value','is_high'])

# Aggregate per cmdb_id:
# sum_anomaly_high: total number of in-window points across all KPIs exceeding P95
# n_kpis_with_high: number of distinct KPIs that had at least one high anomaly
# earliest_high_ts: earliest timestamp among all high anomalies for the service (NaT if none)
# total_points_in_window: total points in window across all KPIs
if not df_win.empty:
    agg_sum = df_win.groupby('cmdb_id').agg(
        sum_anomaly_high=('is_high','sum'),
        n_kpis_with_high=('kpi_name', lambda s: df_win.loc[s.index].loc[df_win.loc[s.index,'is_high']>0,'kpi_name'].nunique()),
        earliest_high_ts=('timestamp', lambda s: df_win.loc[s.index].loc[df_win.loc[s.index,'is_high']>0,'timestamp'].min()),
        total_points_in_window=('timestamp','count')
    ).reset_index()
else:
    # zeroed results for candidates
    agg_sum = pd.DataFrame({
        'cmdb_id': candidates,
        'sum_anomaly_high': [0]*len(candidates),
        'n_kpis_with_high': [0]*len(candidates),
        'earliest_high_ts': [pd.NaT]*len(candidates),
        'total_points_in_window': [0]*len(candidates)
    })

# Ensure all candidate services are present (one row per service)
agg_all = pd.DataFrame({'cmdb_id': candidates})
agg_all = agg_all.merge(agg_sum, on='cmdb_id', how='left')

# Fill missing numeric values with 0 and keep NaT for earliest_high_ts
agg_all['sum_anomaly_high'] = agg_all['sum_anomaly_high'].fillna(0).astype(int)
agg_all['n_kpis_with_high'] = agg_all['n_kpis_with_high'].fillna(0).astype(int)
agg_all['total_points_in_window'] = agg_all['total_points_in_window'].fillna(0).astype(int)
agg_all['earliest_high_ts'] = pd.to_datetime(agg_all['earliest_high_ts'])

# Sort by sum_anomaly_high desc and present compact table (one row per candidate service)
final_table = agg_all.sort_values('sum_anomaly_high', ascending=False).reset_index(drop=True)

final_table

```
Out[5]:
```
Summary (incident window 2024-01-23 06:25:46–06:55:46 UTC):

- ts-travel-service shows the most high anomalies (15 high in-window points across 8 distinct KPIs; earliest high at 06:31 UTC; 200 total points in-window).  
- ts-auth-service, ts-order-service, and ts-route-service each have 14 high anomalies across 7 KPIs (earliest highs: auth 06:28, order 06:29, route 06:29; each with 200 points in-window).  
- ts-train-service also has 14 high anomalies (7 KPIs) with the earliest high at 06:30 UTC and 175 total points in-window.  

Interpretation and next steps:
- ts-travel-service is the top candidate by count of high anomalies and should be investigated first, but ts-auth-service has the earliest anomalous timestamp (06:28 UTC), so it may be an early contributor.  
- Recommend drilling into the specific KPIs that fired for ts-travel and ts-auth around the noted earliest timestamps (06:28–06:31 UTC), and correlate with traces/logs to identify causal interaction.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  sum_anomaly_high  n_kpis_with_high          earliest_high_ts  total_points_in_window
0  ts-travel-service                15                 8 2024-01-23 06:31:00+00:00                     200
1    ts-auth-service                14                 7 2024-01-23 06:28:00+00:00                     200
2   ts-order-service                14                 7 2024-01-23 06:29:00+00:00                     200
3   ts-route-service                14                 7 2024-01-23 06:29:00+00:00                     200
4   ts-train-service                14                 7 2024-01-23 06:30:00+00:00                     175```
```

In [ ]:
# In[6]:

import pandas as pd

# Reuse df_trace if present, else load and parse timestamps
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")

# Candidate services (substrings to match in trace_name)
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']
pattern = '|'.join(candidates)

# Restrict trace rows to those whose trace_name references any candidate service (substring match)
df_trace_cand = df_trace[df_trace['trace_name'].str.contains(pattern, na=False)].copy()

# Compute global P95 and count_total per (cmdb_id, trace_name) using full series (before window filtering)
gb_all = df_trace_cand.groupby(['cmdb_id','trace_name'])['value'].agg(
    count_total='count',
    p95_value=lambda x: x.quantile(0.95)
).reset_index()

# Filter rows to incident window
df_win = df_trace_cand[(df_trace_cand['timestamp'] >= window_start) & (df_trace_cand['timestamp'] <= window_end)].copy()

# Merge global p95 into window rows and mark high anomalies (value > p95)
if not df_win.empty:
    df_win = df_win.merge(gb_all[['cmdb_id','trace_name','p95_value']], on=['cmdb_id','trace_name'], how='left')
    df_win['is_high'] = df_win['value'] > df_win['p95_value']
else:
    # empty consistent frame
    df_win = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95_value','is_high'])

# Aggregate per (cmdb_id, trace_name)
if not df_win.empty:
    agg_win = df_win.groupby(['cmdb_id','trace_name']).agg(
        count_in_window=('value','count'),
        min_in_window=('value','min'),
        max_in_window=('value','max'),
        earliest_ts_in_window=('timestamp','min'),
        anomaly_count_high=('is_high','sum'),
        earliest_anom_high=('timestamp', lambda s: df_win.loc[s.index][df_win.loc[s.index,'is_high']].timestamp.min() if df_win.loc[s.index,'is_high'].any() else pd.NaT)
    ).reset_index()
else:
    agg_win = pd.DataFrame(columns=['cmdb_id','trace_name','count_in_window','min_in_window','max_in_window','earliest_ts_in_window','anomaly_count_high','earliest_anom_high'])

# Merge with global counts/p95
trace_result = gb_all.merge(agg_win, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for numeric fields
trace_result['count_in_window'] = trace_result['count_in_window'].fillna(0).astype(int)
trace_result['min_in_window'] = trace_result['min_in_window']
trace_result['max_in_window'] = trace_result['max_in_window']
trace_result['earliest_ts_in_window'] = pd.to_datetime(trace_result['earliest_ts_in_window'])
trace_result['anomaly_count_high'] = trace_result['anomaly_count_high'].fillna(0).astype(int)
trace_result['earliest_anom_high'] = pd.to_datetime(trace_result['earliest_anom_high'])

# Select and order requested columns
trace_result_table = trace_result[['cmdb_id','trace_name','count_total','p95_value',
                                   'count_in_window','min_in_window','max_in_window',
                                   'earliest_ts_in_window','anomaly_count_high','earliest_anom_high']]

# Sort by anomaly_count_high desc then count_total desc, limit to top 50
trace_result_top50 = trace_result_table.sort_values(['anomaly_count_high','count_total'], ascending=[False, False]).head(50).reset_index(drop=True)

# CROSS-SERVICE SUMMARY:
# Identify window rows that are high, then for each timestamp list distinct candidate services (by substring match) that had a high at that timestamp.
if not df_win.empty:
    df_high = df_win[df_win['is_high']].copy()
    # For safety, determine which candidate substring(s) the trace_name references (may reference multiple; we pick distinct matches)
    def matched_services(trace_name):
        return [s for s in candidates if s in trace_name]
    df_high['matched_services'] = df_high['trace_name'].apply(matched_services)
    # Explode services so we can group by timestamp and service
    df_expl = df_high.explode('matched_services').dropna(subset=['matched_services'])
    # Group by timestamp and collect distinct services with highs at that minute
    cross_summary = df_expl.groupby('timestamp').agg(
        services_with_high = ('matched_services', lambda s: ','.join(sorted(s.unique())))
    ).reset_index().sort_values('timestamp').head(20)
else:
    cross_summary = pd.DataFrame(columns=['timestamp','services_with_high'])

# Display compact results
trace_result_top50, cross_summary

```
Out[6]:
```
summary = (
    "Trace analysis summary (window 2024-01-23 06:25:46–06:55:46 UTC):\n\n"
    "- Many trace groups that reference the candidate services show brief high-value spikes above their global P95 (most groups have 2 high samples in-window).\n"
    "- Notable affected trace groups include root→ts-auth-service (duration_mean, duration_p95, row_count), and many root→ts-order/ts-train/ts-travel traces — each with in-window highs.\n"
    "- High anomalies are clustered in minutes around 06:28–06:33 UTC (e.g., 06:28, 06:29, 06:30, 06:31, 06:32, 06:33) and continue intermittently through ~06:51 UTC. "
    "Multiple candidate services experienced simultaneous highs at several of these minutes (see timestamps 06:28, 06:29, 06:30, 06:31 as examples).\n\n"
    "Conclusion / next steps:\n"
    "- These short, cross-service trace spikes indicate a transient period of increased latency or load affecting multiple services (not isolated to a single service).\n"
    "- Prioritize inspecting trace spans and inter-service calls around 06:28–06:33 UTC (start with auth- and order-related spans) and correlate with request volume and logs to find the root cause."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                  trace_name  count_total   p95_value  count_in_window  min_in_window  max_in_window     earliest_ts_in_window  anomaly_count_high        earliest_anom_high
0              root      trace.to_ts-auth-service.duration_mean           25    0.414734               25       0.264351       0.429080 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:31:00+00:00
1              root       trace.to_ts-auth-service.duration_p95           25    0.988146               25       0.453367       1.226409 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:31:00+00:00
2              root          trace.to_ts-auth-service.row_count           25   82.800000               25      17.000000      86.000000 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:29:00+00:00
3              root     trace.to_ts-order-service.duration_mean           25    1.957010               25       1.626021       2.124615 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:37:00+00:00
4              root      trace.to_ts-order-service.duration_p95           25    2.126341               25       1.735252       2.793937 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:37:00+00:00
..              ...                                         ...          ...         ...              ...            ...            ...                       ...                 ...                       ...
45  ts-seat-service  trace.from_ts-travel-service.duration_mean           25    0.092826               25       0.050489       0.093980 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:43:00+00:00
46  ts-seat-service   trace.from_ts-travel-service.duration_p95           25    0.171100               25       0.071735       0.174956 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:43:00+00:00
47  ts-seat-service      trace.from_ts-travel-service.row_count           25  116.000000               25      16.000000     118.000000 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:38:00+00:00
48  ts-seat-service     trace.to_ts-order-service.duration_mean           25    0.023607               25       0.006709       0.027913 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:28:00+00:00
49  ts-seat-service      trace.to_ts-order-service.duration_p95           25    0.045099               25       0.022629       0.060891 2024-01-23 06:28:00+00:00                   2 2024-01-23 06:28:00+00:00

[50 rows x 10 columns],                    timestamp                                 services_with_high
0  2024-01-23 06:28:00+00:00  ts-order-service,ts-train-service,ts-travel-se...
1  2024-01-23 06:29:00+00:00  ts-auth-service,ts-route-service,ts-train-service
2  2024-01-23 06:30:00+00:00  ts-auth-service,ts-order-service,ts-train-serv...
3  2024-01-23 06:31:00+00:00                                    ts-auth-service
4  2024-01-23 06:32:00+00:00  ts-order-service,ts-route-service,ts-train-ser...
5  2024-01-23 06:33:00+00:00                   ts-auth-service,ts-order-service
6  2024-01-23 06:35:00+00:00                 ts-train-service,ts-travel-service
7  2024-01-23 06:37:00+00:00                                   ts-order-service
8  2024-01-23 06:38:00+00:00  ts-order-service,ts-route-service,ts-train-ser...
9  2024-01-23 06:39:00+00:00                                    ts-auth-service
10 2024-01-23 06:41:00+00:00  ts-route-service,ts-train-service,ts-travel-se...
11 2024-01-23 06:42:00+00:00  ts-auth-service,ts-order-service,ts-route-serv...
12 2024-01-23 06:43:00+00:00  ts-order-service,ts-train-service,ts-travel-se...
13 2024-01-23 06:44:00+00:00                 ts-order-service,ts-travel-service
14 2024-01-23 06:45:00+00:00                 ts-train-service,ts-travel-service
15 2024-01-23 06:46:00+00:00  ts-route-service,ts-train-service,ts-travel-se...
16 2024-01-23 06:47:00+00:00  ts-order-service,ts-route-service,ts-travel-se...
17 2024-01-23 06:48:00+00:00                 ts-order-service,ts-travel-service
18 2024-01-23 06:50:00+00:00  ts-order-service,ts-route-service,ts-train-ser...
19 2024-01-23 06:51:00+00:00                  ts-order-service,ts-route-service)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd

# Reuse df_trace if available, else load and parse timestamps
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Incident window
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")

# Candidate services
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Restrict to trace rows referencing any candidate (substring match)
pattern = '|'.join(candidates)
df_trace_cand = df_trace[df_trace['trace_name'].str.contains(pattern, na=False)].copy()

# Compute global P95 and count_total per (cmdb_id, trace_name) using full series (before window filtering)
gb_all = df_trace_cand.groupby(['cmdb_id','trace_name'])['value'].agg(
    count_total='count',
    p95_value=lambda x: x.quantile(0.95)
).reset_index()

# Filter to incident window and merge p95
df_win = df_trace_cand[(df_trace_cand['timestamp'] >= window_start) & (df_trace_cand['timestamp'] <= window_end)].copy()
df_win = df_win.merge(gb_all[['cmdb_id','trace_name','p95_value']], on=['cmdb_id','trace_name'], how='left')

# Mark high anomalies (value > group's P95)
df_win['is_high'] = df_win['value'] > df_win['p95_value']

# Prepare per-group anomaly aggregates (for later top-group listing)
if not df_win.empty:
    high_agg = df_win[df_win['is_high']].groupby(['cmdb_id','trace_name']).agg(
        anomaly_count_high=('is_high','sum'),
        earliest_anom_high=('timestamp','min')
    ).reset_index()
else:
    high_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count_high','earliest_anom_high'])

win_counts = df_win.groupby(['cmdb_id','trace_name']).agg(
    count_in_window=('value','count'),
    min_in_window=('value','min'),
    max_in_window=('value','max'),
    earliest_ts_in_window=('timestamp','min')
).reset_index()

# Merge group-level results with global p95/count_total
group_table = gb_all.merge(win_counts, on=['cmdb_id','trace_name'], how='left') \
                    .merge(high_agg, on=['cmdb_id','trace_name'], how='left')

group_table['count_in_window'] = group_table['count_in_window'].fillna(0).astype(int)
group_table['anomaly_count_high'] = group_table['anomaly_count_high'].fillna(0).astype(int)
group_table['earliest_anom_high'] = pd.to_datetime(group_table['earliest_anom_high'])
group_table['earliest_ts_in_window'] = pd.to_datetime(group_table['earliest_ts_in_window'])

# SERVICE-LEVEL AGGREGATION
rows = []
for svc in candidates:
    to_pat = f"to_ts-{svc}"
    from_pat = f"from_ts-{svc}"
    # groups with at least one high anomaly referencing svc
    svc_high_rows = df_win[df_win['is_high'] & df_win['trace_name'].str.contains(svc, na=False)].copy()
    total_high_points = int(svc_high_rows.shape[0])
    earliest_high_ts = svc_high_rows['timestamp'].min() if not svc_high_rows.empty else pd.NaT
    # distinct 'to' trace groups with high
    n_to = int(high_agg[high_agg['trace_name'].str.contains(to_pat, na=False)]['trace_name'].nunique()) if not high_agg.empty else 0
    # distinct 'from' trace groups with high
    n_from = int(high_agg[high_agg['trace_name'].str.contains(from_pat, na=False)]['trace_name'].nunique()) if not high_agg.empty else 0
    rows.append({
        'service': svc,
        'n_to_trace_groups_with_high': n_to,
        'n_from_trace_groups_with_high': n_from,
        'total_high_points': total_high_points,
        'earliest_high_ts': earliest_high_ts
    })

services_df = pd.DataFrame(rows).sort_values('total_high_points', ascending=False).reset_index(drop=True)
services_df['earliest_high_ts'] = pd.to_datetime(services_df['earliest_high_ts'])

# For the top service, list up to 10 anomalous trace groups referencing that service
top_service = services_df.iloc[0]['service'] if not services_df.empty else None
if top_service:
    svc_mask = group_table['trace_name'].str.contains(top_service, na=False)
    svc_groups = group_table[svc_mask & (group_table['anomaly_count_high']>0)].copy()
    top_groups = svc_groups[['trace_name','count_total','p95_value','count_in_window','anomaly_count_high','earliest_anom_high']] \
                 .sort_values(['anomaly_count_high','count_total'], ascending=[False, False]).head(10).reset_index(drop=True)
else:
    top_groups = pd.DataFrame(columns=['trace_name','count_total','p95_value','count_in_window','anomaly_count_high','earliest_anom_high'])

# Return compact tables
services_df, top_groups

```
Out[7]:
```
Summary of trace high-anomaly aggregation (window 2024-01-23 06:25:46–06:55:46 UTC):

Service-level totals (sorted by total_high_points):
- ts-travel-service: total_high_points = 69, earliest_high_ts = 2024-01-23 06:28:00 UTC  
- ts-order-service:  total_high_points = 48, earliest_high_ts = 2024-01-23 06:28:00 UTC  
- ts-train-service:  total_high_points = 30, earliest_high_ts = 2024-01-23 06:28:00 UTC  
- ts-route-service:  total_high_points = 29, earliest_high_ts = 2024-01-23 06:29:00 UTC  
- ts-auth-service:   total_high_points = 18, earliest_high_ts = 2024-01-23 06:29:00 UTC  

(Note: the per-service counts of distinct "to" or "from" trace group names with highs are reported as zero in this aggregation output.)

Top service detail — ts-travel-service (anomalous trace groups, up to 10):
- trace.to_ts-travel-service.duration_mean — count_total=25, anomaly_count_high=2, earliest_anom_high=2024-01-23 06:42:00 UTC  
- trace.to_ts-travel-service.duration_p95 — count_total=25, anomaly_count_high=2, earliest_anom_high=2024-01-23 06:41:00 UTC  
- trace.to_ts-travel-service.row_count — count_total=25, anomaly_count_high=2, earliest_anom_high=2024-01-23 06:35:00 UTC  
- trace.to_ts-travel-service.duration_mean (other group) — count_total=25, anomaly_count_high=2, earliest_anom_high=2024-01-23 06:45:00 UTC  
- trace.to_ts-travel-service.duration_p95 (other group) — count_total=25, anomaly_count_high=2, earliest_anom_high=2024-01-23 06:45:00 UTC  
- trace.to_ts-travel-service.duration_mean (other group) — count_total=25, anomaly_count_high=2, earliest_anom_high=2024-01-23 06:41:00 UTC  
- trace.to_ts-travel-service.duration_p95 (other group) — count_total=25, anomaly_count_high=2, earliest_anom_high=2024-01-23 06:41:00 UTC  
- trace.to_ts-travel-service.row_count (other group) — count_total=25, anomaly_count_high=2, earliest_anom_high=2024-01-23 06:47:00 UTC  
- trace.from_ts-travel-service.duration_mean — count_total=25, anomaly_count_high=2, earliest_anom_high=2024-01-23 06:28:00 UTC  
- trace.from_ts-travel-service.duration_p95 — count_total=25, anomaly_count_high=2, earliest_anom_high=2024-01-23 06:28:00 UTC

Key takeaways and next steps:
- ts-travel-service produced the most high-value trace points during the window and should be prioritized for investigation.  
- High anomalies occur across multiple trace metrics (duration_mean, duration_p95, row_count) and are clustered at several minutes (earliest noted at ~06:28 UTC, with additional spikes around 06:35–06:45 UTC).  
- Action: inspect the trace spans and related logs for ts-travel-service at the listed timestamps, and correlate with upstream/downstream services to find the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(             service  n_to_trace_groups_with_high  n_from_trace_groups_with_high  total_high_points          earliest_high_ts
0  ts-travel-service                            0                              0                 69 2024-01-23 06:28:00+00:00
1   ts-order-service                            0                              0                 48 2024-01-23 06:28:00+00:00
2   ts-train-service                            0                              0                 30 2024-01-23 06:28:00+00:00
3   ts-route-service                            0                              0                 29 2024-01-23 06:29:00+00:00
4    ts-auth-service                            0                              0                 18 2024-01-23 06:29:00+00:00,                                    trace_name  count_total  p95_value  count_in_window  anomaly_count_high        earliest_anom_high
0    trace.to_ts-travel-service.duration_mean           25   1.362689               25                   2 2024-01-23 06:42:00+00:00
1     trace.to_ts-travel-service.duration_p95           25   1.622262               25                   2 2024-01-23 06:41:00+00:00
2        trace.to_ts-travel-service.row_count           25  24.800000               25                   2 2024-01-23 06:35:00+00:00
3    trace.to_ts-travel-service.duration_mean           25   0.543701               25                   2 2024-01-23 06:45:00+00:00
4     trace.to_ts-travel-service.duration_p95           25   0.676946               25                   2 2024-01-23 06:45:00+00:00
5    trace.to_ts-travel-service.duration_mean           25   0.083212               25                   2 2024-01-23 06:41:00+00:00
6     trace.to_ts-travel-service.duration_p95           25   0.176954               25                   2 2024-01-23 06:41:00+00:00
7        trace.to_ts-travel-service.row_count           25  26.400000               25                   2 2024-01-23 06:47:00+00:00
8  trace.from_ts-travel-service.duration_mean           25   0.015376               25                   2 2024-01-23 06:28:00+00:00
9   trace.from_ts-travel-service.duration_p95           25   0.041077               25                   2 2024-01-23 06:28:00+00:00)```
```

In [ ]:
# In[8]:

import pandas as pd

# Reuse df_trace if present, else load and parse timestamps
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")

# Restrict to trace rows referencing ts-travel-service (substring match)
pattern = 'ts-travel-service'
df_travel = df_trace[df_trace['trace_name'].str.contains(pattern, na=False)].copy()

# Compute global P95 per trace_name using the full available series (before window filtering)
p95_by_trace = df_travel.groupby('trace_name')['value'].quantile(0.95).rename('p95_value').reset_index()

# Filter to incident window and merge p95 values
df_travel_win = df_travel[(df_travel['timestamp'] >= window_start) & (df_travel['timestamp'] <= window_end)].copy()
df_travel_win = df_travel_win.merge(p95_by_trace, on='trace_name', how='left')

# Select anomalous rows where value > group's P95
df_anom = df_travel_win[df_travel_win['value'] > df_travel_win['p95_value']].copy()

# Add direction column based on trace_name content
def direction_from_name(name):
    if 'to_ts-travel-service' in name:
        return 'to'
    if 'from_ts-travel-service' in name:
        return 'from'
    return 'other'

if not df_anom.empty:
    df_anom['direction'] = df_anom['trace_name'].apply(direction_from_name)
else:
    df_anom['direction'] = pd.Series(dtype='object')

# Prepare anomalous rows output (requested columns), sorted by timestamp asc, limit 50
anom_rows = df_anom[['timestamp','trace_name','value','p95_value','direction']].sort_values('timestamp', ascending=True).head(50)

# Aggregation by direction: count and earliest timestamp
if not df_anom.empty:
    dir_agg = df_anom.groupby('direction').agg(
        anomaly_count=('trace_name','count'),
        earliest_anom_ts=('timestamp','min')
    ).reset_index()
else:
    dir_agg = pd.DataFrame(columns=['direction','anomaly_count','earliest_anom_ts'])

# Display compact results (variables as outputs)
anom_rows, dir_agg

```
Out[8]:
```
Summary (ts-travel-service trace anomalies, window 2024-01-23 06:25:46–06:55:46 UTC):

- Total anomalous trace rows found: 48 (in-window points with value > that trace_name's global P95).  
- Direction breakdown: 24 "from" anomalies and 24 "to" anomalies.  
- Earliest anomalous timestamp: 2024-01-23 06:32:00 UTC.  

Notable patterns from the anomalies:
- Several large row_count spikes for both to_ts-travel-service and from_ts-travel-service at ~06:32, 06:35–06:36 (these exceed their P95).  
- Some duration_p95/duration_mean traces also exceed P95 later in the window (examples around ~06:41–06:52).  

Recommended next steps:
- Start investigation at 06:32 UTC (when both to/from row_count spikes first appear). Correlate these trace spikes with ts-travel-service logs, request volume, and metric KPIs to identify the root cause and affected call paths.

The original code execution output of IPython Kernel is also provided below for reference:

(                     timestamp                                  trace_name        value    p95_value direction
235  2024-01-23 06:32:00+00:00      trace.from_ts-travel-service.row_count  1557.000000  1459.600000      from
239  2024-01-23 06:32:00+00:00        trace.to_ts-travel-service.row_count  1557.000000  1461.200000        to
379  2024-01-23 06:35:00+00:00      trace.from_ts-travel-service.row_count  1523.000000  1459.600000      from
383  2024-01-23 06:35:00+00:00        trace.to_ts-travel-service.row_count  1523.000000  1461.200000        to
427  2024-01-23 06:36:00+00:00      trace.from_ts-travel-service.row_count  1500.000000  1459.600000      from
...                        ...                                         ...          ...          ...       ...
1101 2024-01-23 06:51:00+00:00     trace.to_ts-travel-service.duration_p95     1.597149     1.573867        to
1125 2024-01-23 06:51:00+00:00   trace.from_ts-travel-service.duration_p95     0.166023     0.163034      from
1172 2024-01-23 06:52:00+00:00  trace.from_ts-travel-service.duration_mean     0.086110     0.080544      from
1149 2024-01-23 06:52:00+00:00     trace.to_ts-travel-service.duration_p95     1.594830     1.573867        to
1173 2024-01-23 06:52:00+00:00   trace.from_ts-travel-service.duration_p95     0.163073     0.163034      from

[48 rows x 5 columns],   direction  anomaly_count          earliest_anom_ts
0      from             24 2024-01-23 06:32:00+00:00
1        to             24 2024-01-23 06:32:00+00:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd

# reuse df_trace if available, else load and parse
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Incident window and minute-range filter (UTC)
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")
minute_start = pd.Timestamp("2024-01-23 06:28:00", tz="UTC")
minute_end   = pd.Timestamp("2024-01-23 06:33:00", tz="UTC")

# Candidate service substrings
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1) Compute global P95 per trace_name using the full series (do NOT filter by window)
p95_by_trace = df_trace.groupby('trace_name', dropna=False)['value'].quantile(0.95).rename('p95_value').reset_index()

# 2) Filter rows to the incident window, then to the minute-range, then to trace_names containing any candidate substring
df_win = df_trace[(df_trace['timestamp'] >= window_start) & (df_trace['timestamp'] <= window_end)].copy()
df_minrange = df_win[(df_win['timestamp'] >= minute_start) & (df_win['timestamp'] <= minute_end)].copy()

# restrict to trace_name values that reference any candidate service substring
pattern = '|'.join(candidates)
df_minrange_cand = df_minrange[df_minrange['trace_name'].str.contains(pattern, na=False)].copy()

# 3) Merge global p95 and mark anomalies (value > p95)
df_minrange_cand = df_minrange_cand.merge(p95_by_trace, on='trace_name', how='left')
df_minrange_cand['is_high'] = df_minrange_cand['value'] > df_minrange_cand['p95_value']

# Keep only anomalous rows
df_anom = df_minrange_cand[df_minrange_cand['is_high']].copy()

# 4) Add 'direction' column: 'to' if trace_name contains 'to_ts-<service>', 'from' if contains 'from_ts-<service>', else 'other'
def direction_from_name(name):
    for svc in candidates:
        if f"to_ts-{svc}" in name:
            return 'to'
    for svc in candidates:
        if f"from_ts-{svc}" in name:
            return 'from'
    return 'other'

if not df_anom.empty:
    df_anom['direction'] = df_anom['trace_name'].apply(direction_from_name)
else:
    df_anom['direction'] = pd.Series(dtype='object')

# Select requested columns and sort
anom_rows = df_anom[['timestamp','cmdb_id','trace_name','value','p95_value','direction']] \
            .sort_values(['timestamp','trace_name'], ascending=[True, True]) \
            .head(200) \
            .reset_index(drop=True)

# 5) Per-minute summary for 06:28–06:33: for each minute list distinct candidate services that had any anomalous trace rows at that minute
if not df_anom.empty:
    # determine which candidate services are referenced in each trace_name
    def matched_services_in_name(name):
        return sorted([svc for svc in candidates if svc in name])
    df_anom['matched_services'] = df_anom['trace_name'].apply(matched_services_in_name)
    df_expl = df_anom.explode('matched_services').dropna(subset=['matched_services'])
    per_min_summary = df_expl.groupby('timestamp', sort=True).agg(
        services_with_anomaly = ('matched_services', lambda s: ','.join(sorted(set(s))))
    ).reset_index()
    # ensure only minutes in requested minute range and sort
    per_min_summary = per_min_summary[(per_min_summary['timestamp'] >= minute_start) & (per_min_summary['timestamp'] <= minute_end)] \
                      .sort_values('timestamp').reset_index(drop=True)
else:
    per_min_summary = pd.DataFrame(columns=['timestamp','services_with_anomaly'])

# Display compact outputs
anom_rows, per_min_summary

```
Out[9]:
```
Summary of anomalous trace rows between 06:28:00 and 06:33:00 UTC (incident window 06:25:46–06:55:46):

- Total anomalous trace rows returned in that minute range: 27.  
- Earliest anomalous timestamp observed: 2024-01-23 06:28:00 UTC.  
- Primary symptom: several trace row_count and duration metrics exceeded their global P95 (i.e., short spikes) across multiple services.

Per-minute service-level view (which candidate services had any anomalous traces at each minute):
- 06:28 — ts-order-service, ts-train-service  
- 06:29 — ts-auth-service, ts-train-service  
- 06:30 — ts-auth-service, ts-order-service  
- 06:31 — ts-auth-service, ts-order-service  
- 06:32 — ts-order-service, ts-route-service, ts-train-service (also large row_count spikes for ts-travel observed at ~06:32 in broader window)  
- 06:33 — ts-auth-service, ts-order-service, ts-route-service

Notable examples (brief):
- Large row_count spikes for both to/from traces: ts-travel-service and ts-train-service around 06:32–06:36.  
- Auth-service shows elevated row_count/duration values at ~06:29–06:31.  
- Order-, route-, and train-related traces show intermittent highs during 06:28–06:33.

Recommendation:
- Start root-cause investigation focused on 06:28–06:33 UTC. Prioritize checking trace spans with elevated row_count and duration (auth, order, train, travel), and correlate with logs and request volume to identify whether a traffic surge, backend contention, or an upstream issue caused these short, cross-service spikes.

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp            cmdb_id                               trace_name        value    p95_value direction
0  2024-01-23 06:28:00+00:00               root  trace.to_ts-order-service.duration_mean     1.869478     1.830843     other
1  2024-01-23 06:28:00+00:00   ts-basic-service  trace.to_ts-train-service.duration_mean     0.004943     0.004775     other
2  2024-01-23 06:28:00+00:00   ts-basic-service   trace.to_ts-train-service.duration_p95     0.009356     0.006995     other
3  2024-01-23 06:29:00+00:00    ts-auth-service     trace.from_ts-auth-service.row_count   342.000000   331.200000     other
4  2024-01-23 06:29:00+00:00    ts-auth-service       trace.to_ts-auth-service.row_count   342.000000   327.100000     other
..                       ...                ...                                      ...          ...          ...       ...
22 2024-01-23 06:32:00+00:00   ts-train-service      trace.to_ts-train-service.row_count  2552.000000  2449.800000     other
23 2024-01-23 06:32:00+00:00  ts-travel-service     trace.to_ts-travel-service.row_count  1557.000000  1461.200000     other
24 2024-01-23 06:33:00+00:00               root    trace.to_ts-auth-service.duration_p95     0.993905     0.951697     other
25 2024-01-23 06:33:00+00:00   ts-order-service      trace.to_ts-order-service.row_count   451.000000   441.100000     other
26 2024-01-23 06:33:00+00:00   ts-route-service      trace.to_ts-route-service.row_count  1964.000000  1958.300000     other

[27 rows x 6 columns],                   timestamp                              services_with_anomaly
0 2024-01-23 06:28:00+00:00                  ts-order-service,ts-train-service
1 2024-01-23 06:29:00+00:00                   ts-auth-service,ts-train-service
2 2024-01-23 06:30:00+00:00                   ts-auth-service,ts-order-service
3 2024-01-23 06:31:00+00:00                   ts-auth-service,ts-order-service
4 2024-01-23 06:32:00+00:00  ts-order-service,ts-route-service,ts-train-ser...
5 2024-01-23 06:33:00+00:00  ts-auth-service,ts-order-service,ts-route-service)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd

# Reuse df_trace if present, else load and parse timestamps
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# Compute global P95 per trace_name using the full series (do NOT filter by window)
p95_by_trace = df_trace.groupby('trace_name', dropna=False)['value'].quantile(0.95).rename('p95_value').reset_index()

# Incident window and target minute (UTC)
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")
target_minute = pd.Timestamp("2024-01-23 06:28:00", tz="UTC")

# Filter to incident window, then to the exact target minute
df_win = df_trace[(df_trace['timestamp'] >= window_start) & (df_trace['timestamp'] <= window_end)].copy()
df_minute = df_win[df_win['timestamp'] == target_minute].copy()

# Restrict to trace_name values that reference any candidate service substring
pattern = '|'.join(candidates)
df_minute_cand = df_minute[df_minute['trace_name'].str.contains(pattern, na=False)].copy()

# Merge global p95 and mark anomalies (value > p95)
df_minute_cand = df_minute_cand.merge(p95_by_trace, on='trace_name', how='left')
df_minute_cand['is_high'] = df_minute_cand['value'] > df_minute_cand['p95_value']

# Keep only anomalous rows
df_anom = df_minute_cand[df_minute_cand['is_high']].copy()

# Direction column: 'to' if trace_name contains 'to_ts-<service>' else 'from' if contains 'from_ts-<service>' else 'other'
def direction_from_name(name):
    for svc in candidates:
        if f"to_ts-{svc}" in name:
            return 'to'
    for svc in candidates:
        if f"from_ts-{svc}" in name:
            return 'from'
    return 'other'

if not df_anom.empty:
    df_anom['direction'] = df_anom['trace_name'].apply(direction_from_name)
else:
    df_anom['direction'] = pd.Series(dtype='object')

# Output (1): anomalous rows at 06:28 with requested columns, sorted by trace_name asc, limit 50
anom_rows = df_anom[['timestamp','cmdb_id','trace_name','value','p95_value','direction']].sort_values(['trace_name']).head(50).reset_index(drop=True)

# Output (2): per-service counts of anomalies at 06:28
if not df_anom.empty:
    # Identify matched services per anomalous trace_name
    df_anom['matched_services'] = df_anom['trace_name'].apply(lambda name: [svc for svc in candidates if svc in name])
    df_expl = df_anom.explode('matched_services').dropna(subset=['matched_services'])
    per_service_counts = df_expl.groupby('matched_services').agg(
        anomaly_count_at_06_28=('trace_name','count')
    ).reset_index().rename(columns={'matched_services':'service'})
else:
    per_service_counts = pd.DataFrame({'service': candidates, 'anomaly_count_at_06_28': [0]*len(candidates)})

# Ensure all candidate services appear in the count table (fill zeros if missing)
per_service_counts = pd.DataFrame({'service': candidates}).merge(per_service_counts, on='service', how='left')
per_service_counts['anomaly_count_at_06_28'] = per_service_counts['anomaly_count_at_06_28'].fillna(0).astype(int)

# Display compact results
anom_rows, per_service_counts

```
Out[10]:
```
Summary for 2024-01-23 06:28:00 UTC (trace anomalies):

- Total anomalous trace rows at 06:28:00: 3.
- Affected candidate services:
  - ts-train-service: 2 anomalous rows
    - trace.to_ts-train-service.duration_mean — value 0.004943 > p95 0.004775
    - trace.to_ts-train-service.duration_p95  — value 0.009356 > p95 0.006995
  - ts-order-service: 1 anomalous row
    - trace.to_ts-order-service.duration_mean — value 1.869478 > p95 1.830843
- The other candidate services (ts-auth-service, ts-route-service, ts-travel-service) had no anomalous trace rows at this exact minute.

Recommended next steps:
- Investigate trace spans and logs for ts-train-service and ts-order-service around 06:28:00 UTC (the listed trace names and timestamps), and correlate with request volume and related metrics to determine cause of these short spikes.

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp           cmdb_id                               trace_name     value  p95_value direction
0 2024-01-23 06:28:00+00:00              root  trace.to_ts-order-service.duration_mean  1.869478   1.830843     other
1 2024-01-23 06:28:00+00:00  ts-basic-service  trace.to_ts-train-service.duration_mean  0.004943   0.004775     other
2 2024-01-23 06:28:00+00:00  ts-basic-service   trace.to_ts-train-service.duration_p95  0.009356   0.006995     other,              service  anomaly_count_at_06_28
0    ts-auth-service                       0
1   ts-order-service                       1
2   ts-route-service                       0
3   ts-train-service                       2
4  ts-travel-service                       0)```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np

# Reuse df_metric if available, else load and parse timestamps
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
# ensure timestamp parsed as UTC
if not pd.api.types.is_datetime64_any_dtype(df_metric['timestamp']):
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 06:25:46", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 06:55:46", tz="UTC")

# Candidate services
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Restrict to candidate services (for threshold calc use full series of these services)
df_cand = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (do NOT filter by window)
p95_df = df_cand.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].quantile(0.95).reset_index(name='p95_value')

# 2) Filter points to the incident window
df_win = df_cand[(df_cand['timestamp'] >= window_start) & (df_cand['timestamp'] <= window_end)].copy()

# 3) Merge p95 into window rows and mark high anomalies (value > p95)
df_win = df_win.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')
df_win['is_high'] = df_win['value'] > df_win['p95_value']

# 4) Prepare high-rows dataframe
high_rows = df_win[df_win['is_high']].copy()
# compute relative breach (value / p95_value) ignoring p95_value <= 0 by setting NaN
high_rows['rel_breach'] = np.where(high_rows['p95_value'] > 0, high_rows['value'] / high_rows['p95_value'], np.nan)

# 5) Aggregate per service
# total_high_anomaly_points
total_high = high_rows.groupby('cmdb_id').size().rename('total_high_anomaly_points')
# n_kpis_with_high
n_kpis = high_rows.groupby('cmdb_id')['kpi_name'].nunique().rename('n_kpis_with_high')
# earliest_high_ts
earliest = high_rows.groupby('cmdb_id')['timestamp'].min().rename('earliest_high_ts')
# max_relative_breach
max_rel = high_rows.groupby('cmdb_id')['rel_breach'].max().rename('max_relative_breach')

# Combine into a single summary, ensure all candidate services present
summary = pd.DataFrame({'service': candidates}).set_index('service')
summary = summary.join(total_high.rename_axis('service'), how='left')
summary = summary.join(n_kpis.rename_axis('service'), how='left')
summary = summary.join(earliest.rename_axis('service'), how='left')
summary = summary.join(max_rel.rename_axis('service'), how='left')

# Fill zeros for counts where no anomalies, keep NaT for earliest and NaN for max_relative_breach
summary['total_high_anomaly_points'] = summary['total_high_anomaly_points'].fillna(0).astype(int)
summary['n_kpis_with_high'] = summary['n_kpis_with_high'].fillna(0).astype(int)
summary['earliest_high_ts'] = pd.to_datetime(summary['earliest_high_ts'])
# max_relative_breach leave NaN for services without breaches

# Reset index for display and sort by max_relative_breach desc then total_high_anomaly_points desc
service_summary = summary.reset_index().sort_values(
    by=['max_relative_breach','total_high_anomaly_points'],
    ascending=[False, False],
    na_position='last'
).reset_index(drop=True)

# Keep output compact (one row per candidate service)
service_summary

```
Out[11]:
```
Summary (metrics during 2024-01-23 06:25:46–06:55:46 UTC, candidate services):

- ts-route-service: highest relative breach (max value / p95 ≈ 8.53). 14 high points across 7 KPIs; earliest high at 06:29 UTC. This indicates one or more KPIs spiked far above normal for route service — highest severity by relative magnitude.
- ts-order-service: next-largest relative breach (≈ 2.52). 14 high points across 7 KPIs; earliest high at 06:29 UTC.
- ts-auth-service: 14 high points across 7 KPIs; earliest high at 06:28 UTC; max_relative_breach ≈ 1.09 (smaller relative overshoot but earliest occurrence).
- ts-train-service: 14 high points across 7 KPIs; earliest high at 06:30 UTC; max_relative_breach ≈ 1.08.
- ts-travel-service: most KPIs affected (8) and the largest count of high points (15); earliest high at 06:31 UTC; max_relative_breach ≈ 1.06.

Interpretation and recommended next steps:
- The most concerning signal by severity is ts-route-service (very large relative breach) — investigate the specific KPI(s) that produced that large ratio first.
- ts-travel-service and the others show many high points across multiple KPIs (broad impact). ts-auth-service shows the earliest anomaly (06:28 UTC), so timeline correlation (auth → route/order → travel) may help identify root cause.
- Action: inspect the specific KPI time series and traces/logs for the highlighted services and timestamps (especially route’s offending KPI(s) around 06:29 UTC, and auth around 06:28 UTC), then correlate with traces and request volumes.

The original code execution output of IPython Kernel is also provided below for reference:

service  total_high_anomaly_points  n_kpis_with_high          earliest_high_ts  max_relative_breach
0   ts-route-service                         14                 7 2024-01-23 06:29:00+00:00             8.533825
1   ts-order-service                         14                 7 2024-01-23 06:29:00+00:00             2.515139
2    ts-auth-service                         14                 7 2024-01-23 06:28:00+00:00             1.089233
3   ts-train-service                         14                 7 2024-01-23 06:30:00+00:00             1.076718
4  ts-travel-service                         15                 8 2024-01-23 06:31:00+00:00             1.060706```
```